## Lab 1 - Sampling

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import torch
import matplotlib.pyplot as plt
from diffusers import UNet2DModel

from IPython.display import HTML
from diffusion_utilities import plot_sample
from model import ContextUnet
from train import make_noise_schedule

In [ ]:
print(f"Cuda available: {torch.cuda.is_available()}")

### hyperparams

In [ ]:
timesteps = 500
beta1 = 1e-4
beta2 = 0.02

# network hyperparameters:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
n_features = 64
img_dim = 16  # 16x16 image size
n_channels = 3
n_context = 5

save_dir = "./checkpoints"

In [ ]:
# construct ddpm noise schedule
a_t, b_t, ab_t = make_noise_schedule(timesteps, beta1, beta2, device)

In [ ]:
def denoise_add_noise(x, t, pred_noise, z=None):
    if z is None:
        z = torch.randn_like(x)
    noise = b_t.sqrt()[t] * z
    mean = (x - pred_noise * ((1 - a_t[t]) / (1 - ab_t[t]).sqrt())) / a_t[t].sqrt()
    return mean + noise 

In [ ]:
@torch.no_grad()
def sample_ddpm(model, n_sample, save_rate = 20):
    model.eval()
    samples = torch.randn(n_sample, n_channels, img_dim, img_dim, device=device)
    
    intermediate = []
    for t in tqdm(range(timesteps, 0, -1)):
        time = torch.tensor([t / timesteps]).to(device)

        # sampled noise at this step, only used for t > 1
        z = torch.randn_like(samples) if t > 1 else None

        eps = model(samples, time) # model predicted noise

        if isinstance(eps, tuple):  # for ContextUnet which returns (pred_noise, context)
            samples = denoise_add_noise(samples, t, eps.sample, z)
        else:  # con directly
            samples = denoise_add_noise(samples, t, eps, z)
            
        if t % save_rate == 0 or t == timesteps or t < 8:
            intermediate.append(samples.detach().cpu().numpy())

    intermediate = np.stack(intermediate)
    return samples, intermediate


In [ ]:
!ls -lh checkpoints/context_unet_c/

In [ ]:
# construct model
model = ContextUnet(in_channels=n_channels, n_features=n_features, n_context_features=n_context, img_dim=img_dim).to(
    device
)

selected_epoch = 9
model.load_state_dict(torch.load(f"{save_dir}/context_unet_nc/epoch_{selected_epoch:03d}.pth", map_location=device))



In [ ]:
plt.clf()
samples, intermediate_ddpm = sample_ddpm(model, 32)
animation_ddpm = plot_sample(intermediate_ddpm, 32, 4, save_dir, "ani_run", None, save=False)
HTML(animation_ddpm.to_jshtml())
